# 🌖 chandra_align: GPU Benchmark Notebook for Kaggle

This notebook runs the full GPU-accelerated deep matcher chain (LoFTR + LightGlue + ALIKED) on lunar image pairs for sub-pixel registration benchmarking.

**Target Runtime:** Kaggle Notebook with NVIDIA GPU (Tesla P100/T4)
**Dependencies:** PyTorch CUDA, Kornia, LightGlue, OpenCV, GDAL

In [ ]:
# Cell 1: Environment Setup & GPU Check
import sys
import torch
import numpy as np
import cv2

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Running on CPU — deep matchers will be slow")

# Verify key imports
try:
    import kornia
    print(f"Kornia: {kornia.__version__}")
except ImportError:
    print("Kornia: NOT AVAILABLE")
try:
    import lightglue
    print("LightGlue: AVAILABLE")
except ImportError:
    print("LightGlue: NOT AVAILABLE")

# Add chandra_align to path
sys.path.insert(0, "/kaggle/working")
from chandra_align.matching.deep_matchers import DeepMatcherChain, LightGlueALIKEDMatcher, LoFTRMatcher, ClassicalSIFTMatcher
from chandra_align.matching.sar_optical import SAROpticalGradientMatcher
from chandra_align.preprocessing.sar import refined_lee_filter, frost_filter
print("\n✅ chandra_align imports successful")

In [ ]:
# Cell 2: Sample Data Fetching / Synthetic Pair Generation
import os
import requests
from pathlib import Path

data_dir = Path("/kaggle/working/data")
data_dir.mkdir(exist_ok=True)

# Option A: Try to download real PRADAN sample data (if URLs available)
# Option B: Generate synthetic lunar pairs for benchmarking
def generate_synthetic_lunar_pair():
    """Generate synthetic optical-SAR pair with known affine transform."""
    h, w = 512, 512
    # Create base optical-like image with crater-like features
    ref = np.zeros((h, w), dtype=np.float32)
    np.random.seed(42)
    for _ in range(30):
        cx, cy = np.random.randint(50, w-50), np.random.randint(50, h-50)
        r = np.random.randint(10, 40)
        cv2.circle(ref, (cx, cy), r, np.random.uniform(0.3, 1.0), -1)
    # Add ridges
    for _ in range(15):
        x1, y1 = np.random.randint(0, w), np.random.randint(0, h)
        x2, y2 = np.random.randint(0, w), np.random.randint(0, h)
        cv2.line(ref, (x1, y1), (x2, y2), np.random.uniform(0.2, 0.8), 2)
    ref = cv2.GaussianBlur(ref, (5, 5), 1.0)
    ref = (ref * 255).astype(np.uint8)
    
    # Apply known affine transform (simulate orbital shift)
    M = np.array([
        [1.005, 0.008, 12.3],
        [-0.006, 0.998, -8.7],
    ], dtype=np.float32)
    sec_optical = cv2.warpAffine(ref, M, (w, h))
    
    # Simulate SAR: apply speckle + log-scale + radiometric inversion
    speckle = np.random.gamma(1.0, 1.0, (h, w)).astype(np.float32)
    sec_sar = sec_optical.astype(np.float32) * speckle
    sec_sar = np.log1p(sec_sar)  # log-scale
    sec_sar = 255 - cv2.normalize(sec_sar, None, 0, 255, cv2.NORM_MINMAX)  # inversion
    sec_sar = sec_sar.astype(np.uint8)
    
    # Save
    cv2.imwrite(str(data_dir / "ref_optical.png"), ref)
    cv2.imwrite(str(data_dir / "sec_optical.png"), sec_optical)
    cv2.imwrite(str(data_dir / "sec_sar.png"), sec_sar)
    
    return ref, sec_optical, sec_sar, M

ref, sec_opt, sec_sar, true_M = generate_synthetic_lunar_pair()
print(f"Generated: ref {ref.shape}, sec_optical {sec_opt.shape}, sec_sar {sec_sar.shape}")
print(f"Ground truth affine:\n{true_M}")

In [ ]:
# Cell 3: Run DeepMatcherChain on GPU with Latency Benchmarking
import time
import json

def benchmark_matcher(name, matcher, img_src, img_ref, runs=3):
    """Run matcher multiple times and collect latency stats."""
    times = []
    results = []
    for i in range(runs):
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        start = time.perf_counter()
        res = matcher.match(img_src, img_ref)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        results.append(res)
        print(f"  Run {i+1}: {elapsed:.3f}s | status={res.status} | matches={len(res.pts_src)}")
    
    avg_time = np.mean(times)
    best_res = max(results, key=lambda r: len(r.pts_src) if r.status != "FAILED" else 0)
    
    return {
        "matcher": name,
        "avg_time_sec": round(avg_time, 3),
        "times_sec": [round(t, 3) for t in times],
        "status": best_res.status,
        "matches": len(best_res.pts_src),
        "inliers": int(np.sum(best_res.confidence > 0.5)) if len(best_res.confidence) > 0 else 0
    }

print("=" * 60)
print("GPU DEEP MATCHER BENCHMARK")
print("=" * 60)

# 3a: Optical-Optical (OHRC vs NAC-like) — LightGlue/LoFTR
print("\n📷 Optical ↔ Optical Pair")
chain_opt = DeepMatcherChain()  # Auto-detects GPU matchers
print(f"Available matchers: {[m.name for m in chain_opt.matchers if m.is_available]}")
opt_result = benchmark_matcher("DeepMatcherChain", chain_opt, sec_opt, ref)

# 3b: SAR-Optical Cross-Modal — Phase Congruency / Gradient
print("\n📡 SAR ↔ Optical Cross-Modal Pair")
sar_matcher = SAROpticalGradientMatcher()
sar_result = benchmark_matcher("SAROpticalGradientMatcher", sar_matcher, sec_sar, ref)

# 3c: Classical SIFT fallback (CPU baseline)
print("\n🔍 Classical SIFT (CPU Baseline)")
sift = ClassicalSIFTMatcher()
sift_result = benchmark_matcher("ClassicalSIFTMatcher", sift, sec_opt, ref)

# Summary
print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)
for r in [opt_result, sar_result, sift_result]:
    print(f"{r['matcher']:30s} | {r['avg_time_sec']:6.3f}s | matches={r['matches']:3d} | status={r['status']}")

In [ ]:
# Cell 4: Save COG GeoTIFFs & Render Sub-Pixel Residual Error Vectors
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import matplotlib.pyplot as plt

output_dir = Path("/kaggle/working/outputs")
output_dir.mkdir(exist_ok=True)

# Use best optical result for visualization
best_chain = DeepMatcherChain()
res = best_chain.match(sec_opt, ref)

if res.status != "FAILED" and len(res.pts_src) >= 4:
    # Estimate homography with RANSAC
    H, mask = cv2.findHomography(res.pts_src, res.pts_ref, cv2.RANSAC, 3.0)
    inliers = mask.ravel() == 1
    
    if H is not None and np.sum(inliers) >= 4:
        # Warp secondary to reference
        aligned = cv2.warpPerspective(sec_opt, H, (ref.shape[1], ref.shape[0]))
        
        # Compute residual vectors at inlier locations
        pts_src_in = res.pts_src[inliers]
        pts_ref_in = res.pts_ref[inliers]
        
        pts_src_hom = np.hstack([pts_src_in, np.ones((len(pts_src_in), 1))])
        pts_proj = (H @ pts_src_hom.T).T
        pts_proj = pts_proj[:, :2] / pts_proj[:, 2:]
        
        residuals = pts_proj - pts_ref_in
        rmse = float(np.sqrt(np.mean(np.sum(residuals ** 2, axis=1))))
        
        # Save aligned COG GeoTIFF
        transform = from_bounds(0, 0, ref.shape[1], ref.shape[0], ref.shape[1], ref.shape[0])
        profile = {
            'driver': 'COG',
            'dtype': 'uint8',
            'width': ref.shape[1],
            'height': ref.shape[0],
            'count': 1,
            'crs': CRS.from_epsg(4326),  # Placeholder lunar CRS
            'transform': transform,
            'compress': 'deflate',
            'blocksize': 256,
            'overview_levels': [2, 4, 8, 16]
        }
        
        with rasterio.open(output_dir / "aligned_sec.tif", 'w', **profile) as dst:
            dst.write(aligned, 1)
        
        # Save reference
        with rasterio.open(output_dir / "ref.tif", 'w', **profile) as dst:
            dst.write(ref, 1)
        
        # Save residual vectors as GeoJSON
        import geojson
        features = []
        for i, (src_pt, res_vec) in enumerate(zip(pts_src_in, residuals)):
            features.append(geojson.Feature(
                geometry=geojson.Point((float(src_pt[0]), float(src_pt[1]))),
                properties={
                    "residual_x": float(res_vec[0]),
                    "residual_y": float(res_vec[1]),
                    "residual_magnitude": float(np.linalg.norm(res_vec))
                }
            ))
        fc = geojson.FeatureCollection(features)
        with open(output_dir / "residuals.geojson", 'w') as f:
            geojson.dump(fc, f, indent=2)
        
        # Visualization: residual quiver plot
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(ref, cmap='gray')
        axes[0].set_title('Reference (Optical)')
        axes[0].axis('off')
        
        axes[1].imshow(aligned, cmap='gray')
        axes[1].set_title('Aligned Secondary')
        axes[1].axis('off')
        
        # Quiver plot of residuals
        axes[2].imshow(ref, cmap='gray', alpha=0.5)
        axes[2].quiver(pts_src_in[:, 0], pts_src_in[:, 1], 
                      residuals[:, 0], residuals[:, 1],
                      angles='xy', scale_units='xy', scale=1,
                      color='red', alpha=0.7, width=0.003)
        axes[2].set_title(f'Sub-Pixel Residual Vectors (RMSE={rmse:.3f}px)')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig(output_dir / "residual_visualization.png", dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"✅ COG GeoTIFFs saved to {output_dir}")
        print(f"   aligned_sec.tif, ref.tif")
        print(f"   residuals.geojson ({len(features)} vectors)")
        print(f"   residual_visualization.png")
        print(f"\n📊 RMSE: {rmse:.3f} pixels")
        print(f"   Inliers: {np.sum(inliers)}")
    else:
        print("❌ Homography estimation failed")
else:
    print("❌ Insufficient matches for alignment")

# Save benchmark summary
summary = {
    "optical_optical": opt_result,
    "sar_optical": sar_result,
    "classical_sift": sift_result,
    "rmse_pixels": rmse if 'rmse' in locals() else None,
    "inlier_count": int(np.sum(inliers)) if 'inliers' in locals() else 0
}
with open(output_dir / "benchmark_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n📄 Benchmark summary saved to {output_dir}/benchmark_summary.json")